# Time Series Analysis

This notebook analyzes CRM activity over time and examines how deal creation, sales calls, and deal-closing duration evolve across the observed period.

### Business questions
1. How do deal and call volumes change over time?
2. Do calls and deals move together?
3. How long does it take to close Won and Lost deals?
4. What time window appears most important for converting leads?

> Correlations are descriptive and based on a small number of monthly observations; they should not be interpreted as causal effects.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import (
    colors,
    descriptive_stats,
    cat_stats,
    date_stats,
    plot_distributions,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


## Load Processed Data

In [ ]:
deals = pd.read_pickle(PROCESSED_DIR / 'deals_clean.pkl')
calls = pd.read_pickle(PROCESSED_DIR / 'calls_clean.pkl')
calls = calls[calls['call_start_time'].dt.to_period('M') != '2023-06']

print(f'deals: {deals.shape}')
print(f'calls: {calls.shape}')

## 1. Deal Creation and Call Activity

### 1.1 Monthly Trend

In [ ]:
# Create monthly periods for aggregation

deals['month'] = deals['created_time'].dt.to_period('M')
calls['month'] = calls['call_start_time'].dt.to_period('M')

In [ ]:
# Count deals and calls by month
deals_per_month = deals['month'].value_counts().sort_index()
calls_per_month = calls['month'].value_counts().sort_index()

In [ ]:
deals_per_month

In [ ]:
calls_per_month

In [ ]:
# All deals by month
deals_monthly = deals['month'].value_counts().sort_index().reset_index()
deals_monthly.columns = ['month', 'deals_count']
deals_monthly['month'] = deals_monthly['month'].astype(str)

# All calls by month
calls_monthly = calls['month'].value_counts().sort_index().reset_index()
calls_monthly.columns = ['month', 'calls_count']
calls_monthly['month'] = calls_monthly['month'].astype(str)

In [ ]:
# Buyers by month
mask_buyers = deals['is_buyer']
buyers_monthly = deals[mask_buyers]['month'].value_counts().sort_index().reset_index()
buyers_monthly.columns = ['month', 'buyers_count']
buyers_monthly['month'] = buyers_monthly['month'].astype(str)

# Successful calls by month
mask_success_calls = calls['is_successful']
success_calls_monthly = calls[mask_success_calls]['month'].value_counts().sort_index().reset_index()
success_calls_monthly.columns = ['month', 'success_calls_count']
success_calls_monthly['month'] = success_calls_monthly['month'].astype(str)

In [ ]:
monthly = deals_monthly.merge(calls_monthly, on='month', how='outer').merge(buyers_monthly, on='month', how='outer').merge(success_calls_monthly, on='month', how='outer').sort_values('month')
print("Monthly deal and call statistics")
print(monthly)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Chart 1
ax1 = axes[0]
ax1_twin = ax1.twinx()

sns.lineplot(data=monthly, x='month', y='deals_count', color=colors['accent'], linewidth=2, marker='o', label='Deals', ax=ax1)

sns.lineplot(data=monthly, x='month', y='calls_count', color='#10B981', linewidth=2, marker='o', label='Calls', ax=ax1_twin)

ax1.set_title('Deals and Calls per Month', fontsize=14, fontweight='bold')
ax1.set_ylabel('Deals', color=colors['accent'])
ax1_twin.set_ylabel('Calls', color='#10B981')

ax1.tick_params(axis='y', labelcolor=colors['accent'])
ax1_twin.tick_params(axis='y', labelcolor='#10B981')
ax1.tick_params(axis='x', rotation=45)

# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax1_twin.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left')
ax1_twin.legend_.remove()

# Chart 2
ax2 = axes[1]
ax2_twin = ax2.twinx()

sns.lineplot(data=monthly, x='month', y='buyers_count', color=colors['accent'], linewidth=2, marker='o', label='Buyers', ax=ax2)

sns.lineplot(data=monthly, x='month', y='success_calls_count', color='#10B981', linewidth=2, marker='o', label='Successful Calls', ax=ax2_twin)

ax2.set_title('Buyers and Successful Calls per Month', fontsize=14, fontweight='bold')
ax2.set_xlabel('Month')
ax2.set_ylabel('Buyers', color=colors['accent'])
ax2_twin.set_ylabel('Successful Calls', color='#10B981')

# Axis labels
ax2.tick_params(axis='y', labelcolor=colors['accent'])
ax2_twin.tick_params(axis='y', labelcolor='#10B981')
ax2.tick_params(axis='x', rotation=45)

# Combined legend
h1, l1 = ax2.get_legend_handles_labels()
h2, l2 = ax2_twin.get_legend_handles_labels()
ax2.legend(h1 + h2, l1 + l2, loc='upper left')
ax2_twin.legend_.remove()

plt.tight_layout()
plt.show()

### 1.2 Relationship Between Deal and Call Volumes

In [ ]:
corr_all = monthly['deals_count'].corr(
    monthly['calls_count'],
    method='spearman'
)
print(f'Deals vs calls: r={corr_all:.3f}')

In [ ]:
corr_quality = monthly['buyers_count'].corr(
    monthly['success_calls_count'],
    method='spearman'
)
print(f'Buyers vs successful calls: r={corr_quality:.3f}')

The correlation is calculated from only 12 monthly observations and should therefore be interpreted cautiously.

It describes whether the metrics move together over time; it does **not** establish a causal relationship.

### Key Findings — Monthly Dynamics

**Deals and calls**
- Both metrics increase from Jul 2023 to Apr 2024.
- Activity peaks in **Apr 2024** at roughly **2,900 deals** and **12,700 calls**.
- Deal and call volumes move closely together, which is expected in a sales process where managers respond to incoming lead volume.

**Buyers and successful calls**
- Buyer volume peaks earlier, around **Jan 2024**, while successful-call activity peaks in Apr 2024.
- The higher lead volume in Apr did not produce a proportional increase in buyers.

**Correlation**

| Metric pair | Spearman r | Interpretation |
|---|---:|---|
| Deals vs Calls | 0.937 | Strong positive association |
| Buyers vs Successful Calls | 0.476 | Moderate positive association |

The very strong Deals–Calls relationship is partly structural because calling is part of the sales process. Successful calls alone do not guarantee purchase; offer, price, lead quality, and decision time can also affect conversion.

**Business implication:** scaling lead volume alone was not accompanied by proportional buyer growth. This suggests an opportunity to focus on lead quality and sales-process effectiveness, not only lead acquisition.

## 2. Deal Closing Time and Duration

In [ ]:
# Keep deals with a known valid duration
paid = deals[deals['deal_duration_days'].notna()]

# Summarize deal duration by outcome group
stats = paid.groupby('stage_group')['deal_duration_days'].describe(percentiles=[0.25, 0.50, 0.75, 0.90]).round(2)

print('Deal duration statistics (days)')
stats

In [ ]:
box_data = deals[deals['stage_group'].isin(['Won', 'Lost']) & deals['deal_duration_days'].notna()]

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=box_data, y='stage_group', x='deal_duration_days', hue='stage_group', palette=[colors['accent'], '#E11D48'], legend=False, ax=ax)
ax.set_title("Deals duration by group (days)", fontweight='bold')
ax.set_xlabel('Days from deal creation to closing')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Exclude the extreme upper 1% of deal durations
q99 = deals['deal_duration_days'].quantile(0.99)

paid = deals[deals['duration_status'].isin(['valid', 'same_day']) & deals['deal_duration_days'].notna() & (deals['deal_duration_days'] <= q99)].copy()

stats = paid.groupby('stage_group')['deal_duration_days'].describe(percentiles=[0.25, 0.50, 0.75, 0.90]).round(2)

print('Deal duration statistics (days)')
stats

In [ ]:
# Won deals only
won = deals[
    (deals['stage_group'] == 'Won') &
    (deals['duration_status'].isin(['valid', 'same_day'])) &
    (deals['deal_duration_days'] <= q99)
].copy()

fig, ax = plt.subplots(figsize=(14, 5))

sns.histplot(data=won, x='deal_duration_days', bins=50, color=colors['accent'], ax=ax, kde=True)

ax.set_title('Duration of paid transactions (days)', fontweight='bold')
ax.set_xlabel('Days from deal creation to closing')
ax.set_ylabel('Count of Deals')
plt.tight_layout()
plt.show()

### Key Findings — Deal Duration

Statistics below exclude the extreme upper 1% of deal durations.

| Group | Median | Mean | 75% close within |
|---|---:|---:|---:|
| Won | 17 days | 33 days | 42 days |
| Lost | 4 days | 13 days | 12 days |

Key observations:
- Half of Won deals close within **17 days**.
- 25% of buyers complete payment within about **6 days**, indicating a fast-converting segment.
- Lost deals are identified much sooner, with a median of **4 days**.
- 75% of Won deals close within **42 days**.

**Business implication:** the first six weeks after lead creation appear to be a critical follow-up window for Won deals.